# 1 — Fetch All BankNifty & Nifty Instruments

**Run frequency:** Once a month (or before each expiry cycle)

**What this does:**
1. Logs in to Zerodha KiteConnect via automated TOTP
2. Downloads all NFO instruments from the KiteConnect API
3. Filters BankNifty & Nifty options and futures
4. Calculates current/next weekly and monthly expiry dates
5. Writes everything to Parquet files in `DataFiles/Instruments/`

**Next step:** After this completes, run `2_Fetch_Strikes_Data.ipynb`

In [1]:
# ── Setup: add PROD/ to path so utils/ is importable ──────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [6]:
# ── Step 1: Login to KiteConnect ───────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print("✅ Login successful")

2026-03-11 00:10:35,212 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-11 00:10:35,371 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-11 00:10:35,512 [INFO] utils.kite_helpers — Login successful — request_token: nXJeVlX6yjit3qNU9q0ZkmEnKEkQx7Hu
2026-03-11 00:10:35,680 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [ ]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name="1_Get_Instruments")
print("✅ Spark session ready")

In [ ]:
# ── Step 3: Fetch all instruments from KiteConnect ─────────────────────────
all_instruments = kite.instruments()
print(f"Total instruments fetched: {len(all_instruments)}")
print("Sample:", all_instruments[0])

In [ ]:
# ── Step 4: Filter BankNifty & Nifty instruments ───────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import asc, min as spark_min
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from utils.strike_utils import INSTRUMENT_SCHEMA, EXPIRY_SCHEMA

# Filter by exchange, segment, and name
bnf_options  = [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-OPT' and i['name']=='BANKNIFTY']
nifty_options= [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-OPT' and i['name']=='NIFTY']
bnf_futures  = [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-FUT' and i['name']=='BANKNIFTY']
nifty_futures= [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-FUT' and i['name']=='NIFTY']

# Create Spark DataFrames
BNF_Options_DF     = spark.createDataFrame(bnf_options,   schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
Nifty_Options_DF   = spark.createDataFrame(nifty_options, schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
BNF_Futures_DF     = spark.createDataFrame(bnf_futures,   schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
Nifty_Futures_DF   = spark.createDataFrame(nifty_futures, schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))

print(f"BankNifty Options: {BNF_Options_DF.count()} rows")
print(f"Nifty Options:     {Nifty_Options_DF.count()} rows")
BNF_Options_DF.show(5, truncate=False)

In [ ]:
# ── Step 5: Write instruments to Parquet ────────────────────────────────────
OUTPUT_BASE = 'DataFiles/Instruments'

BNF_Options_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Banknifty_Options')
Nifty_Options_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Nifty_Options')
BNF_Futures_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Banknifty_Futures')
Nifty_Futures_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Nifty_Futures')
print('✅ Instrument Parquet files written')

In [ ]:
# ── Step 6: Calculate expiry dates and write to Parquet ────────────────────
from pyspark.sql.types import StructType, StructField, StringType, DateType

def _min_expiry(df):     return df.agg(spark_min('expiry')).collect()[0][0]
def _next_expiry(df, e): return df.filter(df['expiry'] > e).agg(spark_min('expiry')).collect()[0][0]

bnf_curr_week   = _min_expiry(BNF_Options_DF)
bnf_curr_month  = _min_expiry(BNF_Futures_DF)
bnf_next_week   = _next_expiry(BNF_Options_DF, bnf_curr_week)
bnf_next_month  = _next_expiry(BNF_Futures_DF, bnf_curr_month)

nifty_curr_week  = _min_expiry(Nifty_Options_DF)
nifty_curr_month = _min_expiry(Nifty_Futures_DF)
nifty_next_week  = _next_expiry(Nifty_Options_DF, nifty_curr_week)
nifty_next_month = _next_expiry(Nifty_Futures_DF, nifty_curr_month)

expiry_data = [
    ('BANKNIFTY', bnf_curr_week,   bnf_curr_month,   bnf_next_week,   bnf_next_month),
    ('NIFTY',     nifty_curr_week, nifty_curr_month, nifty_next_week, nifty_next_month),
]

from utils.strike_utils import EXPIRY_SCHEMA
spark.createDataFrame(expiry_data, schema=EXPIRY_SCHEMA)\
     .coalesce(1).write.format('Delta').mode('Overwrite')\
     .parquet(f'{OUTPUT_BASE}/Nifty_Banknifty_Expiries')

print('✅ Expiry Parquet file written')
print(f'BankNifty  current week: {bnf_curr_week}  |  next week: {bnf_next_week}')
print(f'Nifty      current week: {nifty_curr_week}  |  next week: {nifty_next_week}')

---
**Done.** Parquet files are written to `DataFiles/Instruments/`.  
Proceed to `2_Fetch_Strikes_Data.ipynb` to start pulling OHLC data.